# Mens vs Womens Email by Segment

Supplements stage 02 to answer challenge questions 5 and 8 directly:

5. Did the Mens email perform differently from the Womens email across customer segments?
8. Which audience should receive each email, and what data supports that?

**This analysis is exploratory.** It is not in the analysis plan, and results by segment from stages 02 to 04, including both emails' effects in the stage 03 cells, had been seen before it was designed. Holm correction limits false positives within each question, but it does not make the analysis confirmatory.

## Method

- **Data:** all 64,000 customers. No train-test split, because no rule is learned: the decision criteria below are fixed.
- **Question 5:** the six segment splits of stage 02, unchanged. Each regression is `spend ~ mens_email * segment` among emailed customers, with HC3 standard errors. **The interaction is the change in the Mens email's lead over the Womens email.** A split counts as different only if its Holm-adjusted p-value (six tests) is below 0.05.
- **Question 8:** three audiences that do not overlap: bought men's only, women's only, or both. **The default is the Mens email, the winner on average in stage 01. An audience switches to the Womens email only if the Womens lead is significant after Holm correction (three tests).**
- **Worth sending:** each email in each audience is also classified with the rule of 2026-11-06: worth sending if the 95% interval's lower end is above \$0.02, uncertain if only the estimate is, and not worth sending otherwise. **If the Womens email must be used, it is recommended only for audiences where it is worth sending.**
- **Secondary metrics:** visit and conversion are descriptive and decide nothing.
- **Wording:** a difference that is not significant is reported as "not shown", never as "the same".

In [ ]:
import sys

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

sys.path.append("..")
from src.prepare import load_data

ALPHA, BREAK_EVEN = 0.05, 0.008 / 0.40

df = load_data()
df["recent"] = (df["recency"] <= 3).astype(int)
df["online"] = (df["channel"] != "Phone").astype(int)
df["steps"] = df["history_segment"].str[0].astype(int).clip(upper=5) - 1
df["urban"] = df["zip_code"].map({"Rural": 0, "Urban": 1})  # Suburban is left out of this test, as in H5
emailed = df[df["segment"] != "No E-Mail"]

SPLITS = {
    "Recency: 1-3 vs 4-12 months": "recent",
    "Bought women's: yes vs no": "womens",
    "Bought men's: yes vs no": "mens",
    "Channel: Web or Multichannel vs Phone": "online",
    "Past-year spend: per level": "steps",
    "Zip code: Urban vs Rural": "urban",
}
AUDIENCES = ["men's only", "women's only", "both"]
pd.crosstab(df["bought"], df["segment"]).loc[AUDIENCES]

## Question 5: does the Mens email's lead change across segments?

In [ ]:
def lead_interaction(metric, column):
    """Change in the Mens email's lead over the Womens email across one segment split."""
    fit = smf.ols(f"{metric} ~ mens_email * {column}", emailed.dropna(subset=[column])).fit(cov_type="HC3")
    term = f"mens_email:{column}"
    low, high = fit.conf_int().loc[term]
    return {"change_in_lead": fit.params[term], "low": low, "high": high,
            "standard_error": fit.bse[term], "p_value": fit.pvalues[term]}


q5 = pd.DataFrame({name: lead_interaction("spend", column) for name, column in SPLITS.items()}).T
q5["holm_p"] = multipletests(q5["p_value"], method="holm")[1]
q5["min_detectable"] = (norm.ppf(1 - ALPHA / (2 * len(q5))) + norm.ppf(0.8)) * q5["standard_error"]
q5["verdict"] = np.where(q5["holm_p"] < ALPHA, "different", "difference not shown")
q5.round(3)

### Both emails within each segment

Descriptive. A lead that is significant in one segment and not in another does not show that the segments differ; only the table above tests that.

In [ ]:
def classify(estimate, low):
    if low > BREAK_EVEN:
        return "worth sending"
    if estimate > BREAK_EVEN:
        return "uncertain"
    return "not worth sending"


def email_effects(data):
    """Each email against no email, and the Mens email's lead, within one group of customers."""
    fit = smf.ols("spend ~ mens_email + womens_email", data).fit(cov_type="HC3")
    ci = fit.conf_int()
    row = {"customers": len(data)}
    for name, term in [("mens", "mens_email"), ("womens", "womens_email")]:
        row[name] = fit.params[term]
        row[f"{name}_low"], row[f"{name}_high"] = ci.loc[term]
        row[f"{name}_category"] = classify(row[name], row[f"{name}_low"])
    test = fit.t_test("mens_email - womens_email = 0")
    row["lead"] = float(test.effect[0])
    row["lead_low"], row["lead_high"] = test.conf_int()[0]
    row["lead_p"] = float(test.pvalue)
    return row


by_segment = pd.DataFrame({
    (name, int(value)): email_effects(data)
    for name, column in SPLITS.items()
    for value, data in df.dropna(subset=[column]).groupby(column)
}).T
by_segment[["customers", "mens", "womens", "lead", "lead_low", "lead_high"]].astype(float).round(2)

## Question 8: which audience should receive each email?

In [ ]:
q8 = pd.DataFrame({name: email_effects(df[df["bought"] == name]) for name in AUDIENCES}).T
q8["lead_holm_p"] = multipletests(q8["lead_p"].astype(float), method="holm")[1]


def recommend(row):
    if row["lead_holm_p"] < ALPHA:
        return "Mens (shown better)" if row["lead"] > 0 else "Womens (shown better)"
    return "Mens (default, neither shown better)"


q8["email_to_send"] = q8.apply(recommend, axis=1)
q8["womens_if_it_must_be_used"] = np.where(q8["womens_category"] == "worth sending", "yes", "not recommended")

display(q8[["customers", "mens", "mens_low", "mens_high", "mens_category",
            "womens", "womens_low", "womens_high", "womens_category"]])
q8[["lead", "lead_low", "lead_high", "lead_p", "lead_holm_p", "email_to_send", "womens_if_it_must_be_used"]]

## Secondary metrics

Change in the Mens email's lead, in percentage points, with uncorrected intervals. These are descriptive and decide neither question.

In [ ]:
secondary = pd.DataFrame({
    (metric, name): {key: value * 100 for key, value in lead_interaction(metric, column).items()
                     if key in ("change_in_lead", "low", "high")}
    for metric in ["visit", "conversion"]
    for name, column in SPLITS.items()
}).T
secondary.round(2)